# Elastic Fabric Adapter (EFA)

A practical refresher on **Elastic Fabric Adapter (EFA)** — the AWS network interface that gives EC2 instances the low-latency, high-throughput, OS-bypass networking needed for tightly-coupled HPC and large-scale distributed deep-learning training.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Elastic Fabric Adapter (EFA) is an **EC2 network device** that lets applications bypass the operating-system networking stack and talk to the network hardware directly. It is the AWS equivalent of the InfiniBand/RDMA fabrics found in on-prem HPC clusters: it delivers consistently low, low-jitter latency and very high bandwidth between instances in the same cluster placement group.

### What is it?

An EFA is an Elastic Network Adapter (ENA) with an extra capability: an **OS-bypass** path exposed through the **Libfabric** API. Instead of sending traffic through the kernel TCP/IP stack, MPI and NCCL talk to the EFA device via the `efa` Libfabric provider, which uses a custom AWS transport called **SRD (Scalable Reliable Datagram)**. SRD sprays packets across many network paths (equal-cost multipath), tolerates out-of-order delivery, and does its own congestion control and retransmission in hardware — so a single flow can use the full bisection bandwidth of the network.

An EFA still presents a normal ENA interface too, so ordinary TCP/IP traffic (SSH, the AWS API, NFS) keeps working over the same card.

### Why use it?

- **Lower, more predictable latency** for inter-node collective operations (all-reduce, all-gather) than TCP.
- **Higher effective bandwidth** — SRD multipathing avoids the single-flow limits of TCP/ECMP, scaling to hundreds of Gbps per instance (3,200 Gbps on p5.48xlarge across 32 EFA devices).
- **Scales collectives to thousands of GPUs** — distributed training stays communication-efficient as the cluster grows.
- **GPUDirect RDMA** — on p4d/p4de/p5 the EFA reads/writes GPU memory directly, skipping a copy through host RAM.

### When to use it?

- Multi-node **distributed training** (data/tensor/pipeline parallel) where gradient all-reduce dominates step time.
- Tightly-coupled **HPC/MPI** simulations (CFD, weather, molecular dynamics) sensitive to network latency.
- Any job whose scaling efficiency is bottlenecked by inter-node communication.

**When *not* to:** single-node jobs (intra-node NVLink/NVSwitch is used instead), embarrassingly-parallel workloads with little cross-node traffic, or instance types that don't support EFA — there it adds no value.

## Key Features

### Core Capabilities of Elastic Fabric Adapter (EFA)

| Feature | Description | Benefit |
|---------|-------------|---------|
| OS-bypass via Libfabric | Apps use the `efa` provider instead of the kernel TCP stack | Removes kernel/syscall overhead → lower latency & jitter |
| SRD transport | Scalable Reliable Datagram with multipath spraying + HW congestion control | Uses full network bisection bandwidth; resilient to congestion |
| GPUDirect RDMA | NIC DMAs directly to/from GPU HBM (p4d/p4de/p5) | Removes host-memory bounce buffer → higher GPU-to-GPU bandwidth |
| NCCL integration | `aws-ofi-nccl` plugin maps NCCL transport onto Libfabric/EFA | Fast multi-node all-reduce for PyTorch/TensorFlow DDP/FSDP |
| MPI integration | Open MPI / Intel MPI run over the `efa` provider | Drop-in fast fabric for existing HPC codes |
| Multiple EFA devices | High-end instances expose several network cards / EFA interfaces | Aggregate bandwidth (e.g. 32 devices = 3,200 Gbps on p5.48xlarge) |
| Falls back to TCP | Still a normal ENA for non-bypass traffic | Same NIC serves SSH, API calls, and the fabric |

## Architecture Overview

EFA layers an OS-bypass datapath on top of an ENA. Communication libraries (NCCL, MPI) call **Libfabric**, which uses the **`efa` provider** to drive the EFA device over **SRD**, reaching peer instances directly without traversing the kernel TCP/IP stack.

```
   Training / HPC app  (PyTorch DDP/FSDP, Open MPI)
            |
   NCCL  (+ aws-ofi-nccl plugin)   /   MPI
            |
        Libfabric  (libfabric.so)
            |  fi_provider = "efa"
        EFA kernel driver (efa.ko)  ----.   (OS-bypass: kernel out of the datapath)
            |                            |
        EFA device (on the ENA card) <---'
            |  SRD over the VPC network (multipath / ECMP spray)
            v
        Peer EFA device  ->  peer GPU memory (GPUDirect RDMA)

   Cluster placement group keeps instances physically close (low hop count).
```

### Components

1. **EFA device & kernel driver (`efa.ko`)**: the hardware interface plus a thin kernel module that sets up OS-bypass queues; the actual sends/receives skip the kernel.
2. **Libfabric + `efa` provider**: the user-space API (OpenFabrics Interfaces) that applications target; the `efa` provider implements it on EFA using SRD.
3. **`aws-ofi-nccl` plugin**: bridges NVIDIA's NCCL collective library to Libfabric so GPU training uses EFA.
4. **SRD transport**: AWS's reliable-datagram protocol that multipaths packets, reorders at the receiver, and handles congestion/retransmit in hardware.
5. **Cluster placement group**: schedules instances close together in the network to minimize latency and maximize bandwidth between them.

## Installation

### Prerequisites

- An **EFA-capable instance type** in a **cluster placement group**. Examples: `p5.48xlarge`, `p4d.24xlarge`, `p4de.24xlarge`, `trn1.32xlarge`, `c5n.18xlarge`, `c6gn.16xlarge`, `hpc6a.48xlarge`. (`aws ec2 describe-instance-types --query 'InstanceTypes[?NetworkInfo.EfaSupported==`true`].InstanceType'` lists them.)
- A **security group that allows all traffic to and from itself** (a self-referencing inbound + outbound rule). EFA requires this; without it the fabric silently fails.
- The network interface created with `InterfaceType=efa` (launch template, ASG, ParallelCluster, or the EKS device plugin).
- For GPU training: a recent NVIDIA driver, CUDA, and NCCL. The **AWS Deep Learning AMIs / DLC containers ship EFA + `aws-ofi-nccl` preinstalled** — prefer them.

### Installation Steps

Install the EFA software stack (kernel module, Libfabric, and a bundled Open MPI) with the official installer:

In [ ]:
# Run on the instance (Amazon Linux 2/2023 or Ubuntu). Requires sudo.
# The DLAMI / Deep Learning Containers already include this — skip there.

# curl -O https://efa-installer.amazonaws.com/aws-efa-installer-latest.tar.gz
# tar -xf aws-efa-installer-latest.tar.gz
# cd aws-efa-installer
# sudo ./efa_installer.sh -y          # installs efa.ko, libfabric, Open MPI
# # log out/in (or `sudo reboot`) so the kernel module + limits take effect

# Verify the device and provider are present:
# fi_info -p efa -t FI_EP_RDM        # should list the efa provider/endpoints
# /opt/amazon/efa/bin/fi_info -p efa # path when using the bundled libfabric
print("EFA installer steps shown above (commented so the notebook runs anywhere).")

## Basic Usage

### Quick Start Example

You rarely call EFA directly — NCCL and MPI do. The first thing to confirm on a node is that the `efa` Libfabric provider is visible and that the right environment variables are set so NCCL actually uses EFA (and GPUDirect RDMA). The cell below detects EFA if present and prints the recommended environment either way.

In [ ]:
import os
import shutil
import subprocess

def efa_available() -> bool:
    """True if the efa libfabric provider is usable on this host."""
    fi_info = shutil.which("fi_info") or "/opt/amazon/efa/bin/fi_info"
    if not os.path.exists(fi_info) and not shutil.which("fi_info"):
        return False
    try:
        out = subprocess.run([fi_info, "-p", "efa"], capture_output=True, text=True, timeout=10)
        return out.returncode == 0 and "efa" in out.stdout.lower()
    except Exception:
        return False

# Environment that makes NCCL use EFA with GPUDirect RDMA on p4d/p5.
recommended_env = {
    "FI_PROVIDER": "efa",            # force the EFA libfabric provider
    "FI_EFA_USE_DEVICE_RDMA": "1",   # enable GPUDirect RDMA (p4d/p4de/p5)
    "NCCL_PROTO": "simple",          # most robust NCCL protocol over EFA
    "NCCL_DEBUG": "INFO",            # log the transport NCCL selects
    # On the AWS OFI plugin, NCCL discovers EFA automatically; no NCCL_NET needed.
}

if efa_available():
    print("EFA provider detected on this host.")
else:
    print("No EFA here (expected off an EFA instance) — config still applies on one.")

print("\nRecommended environment for NCCL-over-EFA:")
for k, v in recommended_env.items():
    print(f"  export {k}={v}")

### Validating the fabric

Two standard checks confirm EFA is wired up before you launch a real training job:

```bash
# 1) Libfabric-level point-to-point latency/bandwidth between two nodes.
#    On the server node:
fi_pingpong -p efa
#    On the client node (pass the server's IP):
fi_pingpong -p efa <server-ip>

# 2) NCCL all-reduce bandwidth across GPUs/nodes (the number that matters for training).
#    Build nccl-tests against the EFA-aware NCCL, then:
mpirun -n 16 -N 8 --hostfile hosts \
  -x FI_PROVIDER=efa -x FI_EFA_USE_DEVICE_RDMA=1 -x NCCL_DEBUG=INFO \
  ./build/all_reduce_perf -b 8 -e 2G -f 2 -g 1
# Look at the 'busbw' column: it should approach the instance's advertised
# inter-node bandwidth (e.g. ~hundreds of GB/s aggregate on p5 clusters).
```

## Advanced Features

### GPUDirect RDMA, multiple NICs, and tuning the EFA provider

#### GPUDirect RDMA

On p4d/p4de/p5, set `FI_EFA_USE_DEVICE_RDMA=1` so the EFA NIC DMAs straight into GPU HBM, removing the host-memory bounce buffer. This is the single biggest bandwidth win for multi-node GPU training and is enabled by default in recent Libfabric on supported instances.

#### Multiple EFA devices (network cards)

High-end instances expose several physical network cards, each with its own EFA. To use them all you attach one EFA interface **per network card index** (`NetworkCardIndex`) — the `aws-ofi-nccl` plugin then stripes traffic across every device, aggregating bandwidth (e.g. 32 devices × 100 Gbps = 3,200 Gbps on `p5.48xlarge`).

#### Useful EFA / NCCL tuning knobs

In [ ]:
# Reference: environment variables that tune EFA + NCCL for distributed training.
# These are documented values, not live calls — apply them via mpirun -x or the
# container/pod env. Defaults are usually right on the DLAMI; tune only if needed.
efa_nccl_tuning = {
    "FI_EFA_USE_DEVICE_RDMA": "GPUDirect RDMA on/off (1 on p4d/p5)",
    "FI_EFA_FORK_SAFE": "Set 1 if the app fork()s after registering memory",
    "FI_EFA_USE_HUGE_PAGE": "0 to disable huge pages if the host lacks them",
    "NCCL_PROTO": "'simple' is the safest protocol over EFA",
    "NCCL_ALGO": "Ring vs Tree all-reduce; let NCCL auto-tune unless profiling",
    "NCCL_BUFFSIZE": "Per-channel buffer; larger can raise large-message bandwidth",
    "NCCL_SOCKET_IFNAME": "Bootstrap iface (e.g. 'eth0'); EFA data path is separate",
    "RDMAV_FORK_SAFE": "1 for fork-safety in rdma-core",
}
for k, v in efa_nccl_tuning.items():
    print(f"{k:28s} -> {v}")

## Use Cases

### Real-world Applications of Elastic Fabric Adapter (EFA)

#### Use Case 1: Large-language-model pre-training

- **Context**: Train a multi-billion-parameter model across many `p5.48xlarge` nodes with FSDP/tensor parallelism.
- **Implementation**: Cluster placement group + EFA + GPUDirect RDMA; PyTorch with NCCL and the `aws-ofi-nccl` plugin; multiple EFA devices per node for full bandwidth.
- **Results**: Gradient all-reduce stays cheap as the cluster scales, keeping GPU utilization (MFU) high instead of stalling on communication.

#### Use Case 2: HPC simulation (CFD / weather / molecular dynamics)

- **Context**: A latency-sensitive MPI code where every step exchanges halo data between ranks.
- **Implementation**: AWS ParallelCluster provisions an EFA-enabled compute fleet; Open MPI uses the `efa` provider over SRD.
- **Results**: Near-on-prem-InfiniBand scaling efficiency without owning a cluster.

#### Use Case 3: Distributed fine-tuning / RLHF on a managed platform

- **Context**: Multi-node fine-tuning launched through SageMaker training jobs.
- **Implementation**: An EFA-enabled instance type + a DLC image with EFA/NCCL preinstalled; SageMaker wires the placement group and interfaces.
- **Results**: Faster epochs from efficient cross-node all-reduce with minimal infrastructure setup.

## Best Practices

### Recommended Practices for Elastic Fabric Adapter (EFA)

1. **Always use a cluster placement group** — EFA's latency benefit depends on instances being physically close; without it you lose most of the gain.
2. **Use a self-referencing security group** that allows *all* traffic inbound and outbound to itself. This is the most common reason EFA "doesn't work."
3. **Start from the AWS DLAMI / Deep Learning Containers** — they ship a tested EFA + Libfabric + `aws-ofi-nccl` + NCCL stack, so you avoid version-mismatch debugging.
4. **Enable GPUDirect RDMA** (`FI_EFA_USE_DEVICE_RDMA=1`) on p4d/p5 and attach **all** EFA devices the instance offers to get full bandwidth.
5. **Validate before you scale** — run `fi_pingpong` and `nccl-tests all_reduce_perf` and confirm `busbw` is near the advertised number *before* launching a long, expensive training run.

## Common Pitfalls

### What to Avoid When Using Elastic Fabric Adapter (EFA)

1. **Security group not self-referencing** — EFA needs an SG rule allowing all traffic to/from the same SG; miss it and traffic silently falls back to TCP or fails.
2. **Forgetting the placement group** — instances scattered across the AZ get high, variable latency and EFA underdelivers.
3. **Version skew** between the EFA installer, Libfabric, the kernel module, NCCL, and `aws-ofi-nccl` — mismatches cause crashes or silent TCP fallback. Use the DLAMI/DLC pinned stack.
4. **Not verifying EFA is actually used** — set `NCCL_DEBUG=INFO` and confirm NCCL logs the `AWS Libfabric`/`ofi` transport; otherwise you may be running over slow sockets and never notice.
5. **Expecting EFA to help single-node jobs** — intra-node GPU traffic goes over NVLink/NVSwitch; EFA only matters between instances.

## Performance Optimization

### Optimizing Elastic Fabric Adapter (EFA) for Production

#### Configuration Tuning

Key levers, roughly in order of impact:

- **GPUDirect RDMA** (`FI_EFA_USE_DEVICE_RDMA=1`): removes the host-memory copy on supported instances — the biggest bandwidth win.
- **All EFA devices attached**: multi-NIC instances need one EFA per network card to reach advertised bandwidth.
- **Cluster placement group**: minimizes hop count → lower latency for collectives.
- **NCCL protocol/algorithm** (`NCCL_PROTO`, `NCCL_ALGO`): `simple` is the safe default; let NCCL auto-tune ring vs tree unless you are profiling specific message sizes.
- **Overlap communication with compute**: gradient bucketing (DDP `bucket_cap_mb`) and FSDP prefetch hide all-reduce latency behind backprop.

The cell below shows how to confirm — programmatically — that NCCL chose the EFA/OFI transport, which is the check that catches silent TCP fallback.

In [ ]:
# Parse NCCL_DEBUG=INFO logs to confirm EFA (the AWS OFI plugin) was selected.
# In practice you'd capture stderr from your training launch; here we demo the check.
sample_nccl_log = '''
NCCL INFO NET/OFI Selected Provider is efa
NCCL INFO NET/OFI Using aws-ofi-nccl 1.9.0
NCCL INFO Channel 00 : 0[0] -> 1[0] via NET/OFI/0
'''

def using_efa(nccl_stderr: str) -> bool:
    low = nccl_stderr.lower()
    good = "net/ofi" in low and "efa" in low
    fell_back = "net/socket" in low  # TCP sockets => EFA NOT in use
    return good and not fell_back

if using_efa(sample_nccl_log):
    print("NCCL is using EFA via the AWS OFI plugin (good).")
else:
    print("WARNING: NCCL appears to be on TCP sockets — EFA is NOT being used.")

## Production Deployment

### Deploying Elastic Fabric Adapter (EFA) in Production

You don't "deploy EFA" — you provision instances *with* EFA interfaces and a placement group, then run your training/HPC workload on them. Below are the three common paths.

#### Launch template with an EFA interface (Terraform)

```hcl
resource "aws_placement_group" "cluster" {
  name     = "ml-cluster"
  strategy = "cluster"
}

resource "aws_launch_template" "efa" {
  instance_type = "p5.48xlarge"
  image_id      = data.aws_ami.dlami.id

  network_interfaces {
    interface_type              = "efa"   # <-- makes this an EFA interface
    device_index                = 0
    network_card_index          = 0
    security_groups             = [aws_security_group.efa.id]
    delete_on_termination       = true
  }
  placement { group_name = aws_placement_group.cluster.name }
}
```

#### Self-referencing security group (required by EFA)

```hcl
resource "aws_security_group" "efa" { name = "efa-sg"  vpc_id = var.vpc_id }

resource "aws_security_group_rule" "efa_self_in" {
  type = "ingress"  from_port = 0  to_port = 0  protocol = "-1"
  security_group_id = aws_security_group.efa.id
  source_security_group_id = aws_security_group.efa.id   # all traffic from itself
}
resource "aws_security_group_rule" "efa_self_out" {
  type = "egress"   from_port = 0  to_port = 0  protocol = "-1"
  security_group_id = aws_security_group.efa.id
  source_security_group_id = aws_security_group.efa.id
}
```

#### Kubernetes (EKS) with the EFA device plugin

```yaml
# Install the device plugin (DaemonSet) so EFA shows up as a schedulable resource:
#   helm repo add eks https://aws.github.io/eks-charts
#   helm install efa eks/aws-efa-k8s-device-plugin -n kube-system
apiVersion: v1
kind: Pod
metadata:
  name: efa-trainer
spec:
  containers:
    - name: trainer
      image: <account>.dkr.ecr.us-east-1.amazonaws.com/training:latest
      resources:
        limits:
          nvidia.com/gpu: 8
          vpc.amazonaws.com/efa: 32   # request all EFA devices on a p5.48xlarge
```

> Tip: AWS ParallelCluster and SageMaker can configure the placement group, EFA interfaces, and security group for you — prefer them over hand-rolling for HPC and managed training.

## Monitoring and Observability

### Monitoring Elastic Fabric Adapter (EFA) in Production

#### Key Metrics to Track

The EFA driver exposes hardware counters (readable under `/sys/class/infiniband/*/ports/1/hw_counters/` and via the `ethtool -S` ENA stats). Watch:

- **`rdma_read_bytes` / `rdma_write_bytes`**: data moved over the fabric — your throughput signal.
- **`rx_drops` / `tx_drops`** and **`rdma_read_resp_err` / retransmit counters**: rising values indicate congestion or a misconfiguration.
- **NCCL `busbw` (bus bandwidth)** from periodic `all_reduce_perf` runs: the end-to-end metric that actually predicts training step time.
- **GPU utilization / MFU**: if GPUs idle while comms run, the fabric (or its config) is the bottleneck.

#### Logging Best Practices

- Always launch training with **`NCCL_DEBUG=INFO`** (at least once) and archive the log — it records which transport/provider was chosen and the topology.
- Ship EFA `hw_counters` to **CloudWatch** (CloudWatch agent custom metrics) so you can alarm on drop/retransmit spikes.
- Correlate **per-step time** with comms counters to catch a node whose EFA quietly fell back to TCP mid-fleet.
- Record the exact **EFA installer / Libfabric / NCCL / aws-ofi-nccl versions** with each run for reproducibility.

## Troubleshooting

### Common Issues with Elastic Fabric Adapter (EFA)

#### Issue 1: `fi_info -p efa` returns nothing

**Symptoms**: No EFA provider listed; NCCL falls back to sockets.

**Cause**: EFA installer/kernel module not loaded, or the instance was launched without an EFA interface.

**Solution**: Re-run `sudo ./efa_installer.sh -y` and reboot; confirm `lsmod | grep efa` shows `efa`; confirm the ENI was created with `InterfaceType=efa`.

#### Issue 2: NCCL works but is slow / uses TCP

**Symptoms**: `NCCL_DEBUG=INFO` shows `NET/Socket` instead of `NET/OFI`; low `busbw`.

**Cause**: `aws-ofi-nccl` plugin missing/mismatched, or the security group isn't self-referencing so the fabric can't connect.

**Solution**: Use the DLAMI/DLC stack; add the self-referencing SG rule; set `FI_PROVIDER=efa` and re-check the log for `NET/OFI ... efa`.

#### Issue 3: `ibv_fork_init` / fork-related crashes or memory-registration errors

**Symptoms**: Segfaults or registration failures when the app `fork()`s (common with some data loaders).

**Cause**: Memory registered for RDMA is unsafe across `fork()` without fork-safety enabled.

**Solution**: Export `FI_EFA_FORK_SAFE=1` (and `RDMAV_FORK_SAFE=1`); avoid forking after registering buffers where possible.

## Comparison with Alternatives

### How Elastic Fabric Adapter (EFA) Compares to Other Solutions

| Dimension | EFA | Standard ENA / TCP | On-prem InfiniBand | NVLink / NVSwitch |
|-----------|-----|--------------------|--------------------|--------------------|
| Scope | Inter-node, AWS | Inter-node, AWS | Inter-node, on-prem | Intra-node only |
| Datapath | OS-bypass (Libfabric/SRD) | Kernel TCP/IP | OS-bypass (verbs) | GPU interconnect |
| Latency | Low, low-jitter | Higher, variable | Very low | Lowest (on-board) |
| Single-flow BW | Full (multipath SRD) | Limited by single path | High | Extremely high |
| Ops burden | Managed by AWS | Managed by AWS | You own the fabric | None (in the box) |
| Best for | Multi-node HPC/ML on AWS | General networking | On-prem HPC clusters | GPU-to-GPU in one server |

### When to Choose This Tool

Choose **EFA** when:

- You run **multi-node** distributed training or MPI HPC **on AWS** and inter-node communication limits scaling.
- You want InfiniBand-class fabric performance **without owning hardware**.
- Your instance type supports it and you can use a **cluster placement group**.

Prefer **NVLink/NVSwitch** (automatic, intra-node) for single-server multi-GPU jobs, **standard ENA/TCP** when the workload isn't communication-bound, and **on-prem InfiniBand** only when you're not on AWS.

## Resources

### Official Documentation

- EFA overview (EC2 User Guide): https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/efa.html
- Getting started / installer steps: https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/efa-start.html
- Scalable Reliable Datagram (SRD) — "A Cloud-Optimized Transport Protocol" (IEEE Micro): https://ieeexplore.ieee.org/document/9167399
- Libfabric (OpenFabrics Interfaces): https://ofiwg.github.io/libfabric/

### Tutorials and Guides

- aws-ofi-nccl plugin (GitHub): https://github.com/aws/aws-ofi-nccl
- NCCL tests (all_reduce_perf): https://github.com/NVIDIA/nccl-tests
- AWS ParallelCluster with EFA: https://docs.aws.amazon.com/parallelcluster/latest/ug/efa-v3.html
- EFA-enabled distributed training on SageMaker: https://docs.aws.amazon.com/sagemaker/latest/dg/distributed-training.html

### Community Resources

- aws-efa-k8s-device-plugin (EKS): https://github.com/aws/eks-charts/tree/master/stable/aws-efa-k8s-device-plugin
- AWS HPC blog: https://aws.amazon.com/blogs/hpc/
- AWS re:Post (EFA Q&A): https://repost.aws/tags/questions

### Related Technologies

- NCCL (NVIDIA Collective Communications Library) — the collective layer EFA accelerates
- GPUDirect RDMA — NIC-to-GPU-memory transfers used by EFA on p4d/p5
- AWS Deep Learning AMIs / Deep Learning Containers — preconfigured EFA + NCCL stacks